In [1]:
# 1 - Installing the necessary libraries

# Installing transformers library from Hugging Face (for the chatbot model)
!pip install transformers

# Installing gradio (for building the chatbot's web interface)
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.1 MB/s eta 0:00:00


In [4]:
# 2 - Setting up the chatbot

# Importing the pipeline from transformers
from transformers import pipeline  # for text generation
import gradio as gr  # for creating the web interface

# Loading the BlenderBot model for text-generation task
chatbot = pipeline('text-generation', model='facebook/blenderbot-400M-distill')

# Confirming the model has loaded
print("Chatbot model loaded successfully!")

pytorch_model.bin:   0%|          | 0.00/730M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/730M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/16.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/310k [00:00<?, ?B/s]

Device set to use cpu


Chatbot model loaded successfully!


In [7]:
# 3- Simulating a Conversation (2025 way)
# Since we removed Conversation() because it's an oldn no longer existing class, we're going to:
# Manually build the dialogue.
# Make the chatbot reply step by step.

# Creating a conversation history list to store the full dialogue
history = []

# Step 1 - Starting the conversation with a greeting
user_input = "Hi, how are you?"  # First message from the user

# Step 2 - Adding the user input to the history
history.append(f"User: {user_input}")

# Step 3 - Preparing the conversation string for the model
conversation_text = "\n".join(history)  # Joining all previous exchanges

# Step 4 - Asking the chatbot to generate a reply
response = chatbot(conversation_text, max_new_tokens=50)[0]['generated_text']

# Step 5 - Saving the chatbot's response to the history
history.append(f"Bot: {response}")

# Step 6 - Showing the conversation so far
print("\n".join(history))

User: Hi, how are you?
Bot: User: Hi, how are you? WAYYYYES WESS.


In [9]:
# 4- Creating a Gradio Interface

# Step 1 - Preparing the message and response history
message_list = []   # to store user messages
response_list = []  # to store bot responses

# Step 2 - Defining the chatbot function for Gradio
def mini_chatbot(message, history):
    # Adding the new user message to the history
    message_list.append(message)

    # Preparing the conversation prompt
    conversation_text = ""
    for m, r in zip(message_list, response_list):
        conversation_text += f"User: {m}\nBot: {r}\n"
    conversation_text += f"User: {message}\nBot:"  # <-- tell model it's time to answer

    # Generating the response
    response = chatbot(conversation_text, max_new_tokens=50)[0]['generated_text']

    # Extracting only the new bot response (removing the prompt part)
    reply = response[len(conversation_text):].strip()

    # Saving the reply to the response list
    response_list.append(reply)

    # Returning the reply to Gradio
    return reply

# Step 3 - Creating the chatbot interface
demo_chatbot = gr.ChatInterface(
    fn=mini_chatbot,  # the chatbot logic
    title="BlenderBot Mini",
    description="A simple chatbot using Facebook BlenderBot + Gradio (fixed version)"
)


/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:334: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


In [10]:
# 5- Launch your chatbot:

# Step 4 - Launching the chatbot
demo_chatbot.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://893eed53cf02f2a6ab.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#Bonus

### Making the Bot Smarter and Less Repetitive

The problem with raw text-generation pipelines is:
They sometimes repeat the user input.
They produce random or short replies.
They don't always feel "chatbot-ish".

- We can fix this by:

Improving the prompt formatting.

Giving it a system message (optional).

Controlling the generation parameters like temperature, top_p, and max_new_tokens.

In [11]:
# Improved Version:


# Resetting histories
message_list = []
response_list = []

# Defining the improved chatbot function
def mini_chatbot(message, history):
    # Adding the user message to the message list
    message_list.append(message)

    # OPTIONAL: Adding a system instruction to make the bot act more like a friendly assistant
    system_prompt = "The following is a conversation between a helpful AI assistant and a human. The AI is friendly, concise, and avoids repeating the user input.\n\n"

    # Rebuilding the conversation
    conversation_text = system_prompt
    for m, r in zip(message_list, response_list):
        conversation_text += f"User: {m}\nBot: {r}\n"
    conversation_text += f"User: {message}\nBot:"  # <-- ask the bot to answer

    # Generating the reply with better control
    response = chatbot(
        conversation_text,
        max_new_tokens=60,      # longer responses
        temperature=0.7,         # more creative but not too random
        top_p=0.9                # nucleus sampling
    )[0]['generated_text']

    # Extracting the bot reply
    reply = response[len(conversation_text):].strip()

    # Appending the reply to the response list
    response_list.append(reply)

    # Returning the reply for Gradio
    return reply

# Creating the Gradio interface
demo_chatbot = gr.ChatInterface(
    fn=mini_chatbot,
    title="💬 BlenderBot Mini (Enhanced)",
    description="A smarter version of BlenderBot using transformers + gradio"
)

# Launching the chatbot
demo_chatbot.launch()

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:334: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8934bdeb727b26443b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#I don't see improvment in Chatbot's answers.